# Early warning signals (EWS) on monthly NDVI

Dakos et al. (2012) workflow, on monthly max-NDVI instead of yearly:

1. Load and regularise to monthly (same as the breakpoint notebook)
2. **Remove the seasonal cycle** with STL — essential for monthly data: otherwise AR1 and variance mostly
   measure the yearly cycle, not resilience
3. Find the **loss breakpoints** (≥ 0.2 drop in 12 months that persists ≥ 2 years, same rule as before)
4. For points with a loss: take only the segment **before** the drop (an abrupt step inside the window would
   create fake rising variance/AR1). Points without any breakpoint are used as **controls**.
5. Gaussian detrending → residuals → rolling AR1, SD, skewness, kurtosis
6. Kendall τ of each indicator over time + significance with **AR(1) surrogates**
7. Compare: how often do pre-loss points show rising AR1/SD (hits) vs. the controls (false alarms)?

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau
from scipy.ndimage import gaussian_filter1d
from statsmodels.tsa.seasonal import STL

## Config

In [ ]:
CONFIG = {
    "input_csvs": {
        "base": "/Users/dunyaschotman/Documents/AES_Msc_MDP/code/NDVI_Monthly_data/monthly_max_NDVI_per_point.csv",
        },
    "output_dir": "ews_output_monthly",
    "max_gap_months": 2,         # interior gaps up to this many months are linearly interpolated
    "min_completeness": 0.6,     # drop a point if less than this fraction of months has data
    "stl_trend_months": 13,      # STL trend length for the breakpoint detection (same as breakpoint notebook)

    # Breakpoints (same rule as the breakpoint notebook)
    "bp_threshold": 0.25,         # NDVI drop of at least ...
    "bp_window_months": 12,      # ... within this many months
    "persist_years": 2,          # and the low level must hold this long

    # EWS
    "min_segment_months": 60,    # pre-loss segment must be at least this long (5 years) to compute EWS
    "max_filled_frac": 0.25,     # skip segments with more than this fraction of interpolated months
                                 # (interpolation creates fake autocorrelation -> inflates AR1)
    "bandwidth_frac": 0.2,       # Gaussian detrending bandwidth, fraction of segment length
    "window_frac": 0.5,          # rolling window, fraction of segment length (Dakos default)
    "n_surrogates": 500,         # raise to ~1000 for final results
    "alpha": 0.05,               # significance level
    "random_seed": 0,
    "max_controls": 300,         # random sample of control points for the surrogate test (runtime)
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
rng = np.random.default_rng(CONFIG["random_seed"])

traj_labels = {1: 'Stable interior', 2: 'Stable edge', 3: 'Monotonic loss',
               4: 'Monotonic gain', 5: 'Ambiguous/high turnover'}

## 1. Load and regularise to monthly

In [ ]:
def load_long_csv(path):
    df = pd.read_csv(path).rename(columns={"ndvi_max": "ndvi"})
    df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
    for col in ["traj", "n_images_that_month"]:     # optional columns, not in every export
        if col not in df.columns:
            df[col] = np.nan
    return df[["point_id", "date", "ndvi", "traj", "lon", "lat", "n_images_that_month"]]


def regularize_to_monthly(df, max_gap_months=2, min_completeness=0.6):
    monthly = df.pivot_table(index="date", columns="point_id", values="ndvi").resample("MS").mean()
    observed = monthly.notna()                                   # True = real observation
    interpolated = monthly.interpolate(method="linear", limit=max_gap_months, limit_area="inside")

    completeness = interpolated.notna().mean()
    keep = completeness[completeness >= min_completeness].index
    dropped = completeness[completeness < min_completeness].index.tolist()

    cleaned = interpolated[keep].ffill().bfill()                 # STL can't handle NaNs
    return cleaned, observed[keep], dropped


df = pd.concat([load_long_csv(path).assign(source=name)
                for name, path in CONFIG["input_csvs"].items()], ignore_index=True)
df["point_id"] = df["source"] + "_" + df["point_id"].astype(str)   # unique IDs across the files
print(df.groupby("source")["point_id"].nunique())                    # points per file

monthly, observed, dropped = regularize_to_monthly(df, CONFIG["max_gap_months"], CONFIG["min_completeness"])
point_traj = df.groupby("point_id")["traj"].first()
n_images = df.pivot_table(index="date", columns="point_id", values="n_images_that_month") \
             .resample("MS").mean().reindex(monthly.index)

print(f"{monthly.shape[1]} points kept, {len(dropped)} dropped (too incomplete)")
print(f"period: {monthly.index.min():%Y-%m} to {monthly.index.max():%Y-%m}")

## 2. STL: trend (for breakpoints) and deseasonalised series (for EWS)

* **trend** (13-month smoother) → breakpoint detection, exactly as in the breakpoint notebook
* **deseasonalised** = observed − seasonal → input for the EWS. Not the trend: the trend smoother would remove
  exactly the month-to-month fluctuations that the EWS indicators need.

In [ ]:
trends, deseason = {}, {}
for pid in monthly.columns:
    s = monthly[pid]
    res = STL(s, period=12, trend=CONFIG["stl_trend_months"], robust=True).fit()
    trends[pid] = res.trend
    deseason[pid] = s - res.seasonal
print(f"STL done for {len(trends)} points")

## 3. Loss breakpoints (same rule as the breakpoint notebook)

In [ ]:
def find_breakpoints(trend, threshold, window, persist_years):
    persist = int(round(12 * persist_years))
    trend = trend.iloc[6:-6]                                   # skip STL edge effects
    delta = trend.shift(-window) - trend
    sign = np.sign(delta).where(delta.abs() >= threshold, 0).fillna(0)
    run_id = (sign != sign.shift()).cumsum()

    breakpoints, rejected = [], None
    for _, run in sign.groupby(run_id):
        if run.iloc[0] == 0:
            continue
        t0 = delta.loc[run.index].abs().idxmax()
        t1 = t0 + pd.DateOffset(months=window)
        kind = "rise" if delta.loc[t0] > 0 else "drop"
        level_before = trend.loc[t0 - pd.DateOffset(months=12):t0].median()
        if rejected is not None and rejected[0] != kind and t0 <= rejected[2] + pd.DateOffset(months=persist):
            level_before = rejected[1]
        after = trend.loc[t1:t1 + pd.DateOffset(months=persist)]
        if len(after) <= persist:
            continue
        shift = (after - level_before) if kind == "rise" else (level_before - after)
        if (shift >= threshold).all():
            breakpoints.append({"type": kind, "window_start": t0, "window_end": t1,
                                "magnitude": round(trend.loc[t1] - trend.loc[t0], 3)})
            rejected = None
        else:
            rejected = (kind, level_before, t1)
    return breakpoints


bps = {pid: find_breakpoints(trends[pid], CONFIG["bp_threshold"], CONFIG["bp_window_months"],
                             CONFIG["persist_years"]) for pid in trends}
loss_points = {pid: [b for b in bp if b["type"] == "drop"][0] for pid, bp in bps.items()
               if any(b["type"] == "drop" for b in bp)}     # first loss per point
control_points = [pid for pid, bp in bps.items() if len(bp) == 0]
print(f"{len(loss_points)} points with a loss breakpoint, {len(control_points)} control points (no breakpoint)")

## 4. EWS functions

* **Segment:** for a loss point, everything *before* the start of the 12-month drop window. For a control point,
  the whole series.
* **Detrending:** Gaussian kernel (as in the `earlywarnings` R package), indicators on the residuals.
* **Indicators** in a rolling window of 50% of the segment: AR1 (lag-1 autocorrelation), SD, skewness, kurtosis.
* **Trend:** Kendall τ of each indicator against time. Critical slowing down predicts **rising AR1 and SD**.
* **Significance — AR(1) surrogates:** for each segment we simulate many series with the *same* lag-1
  autocorrelation and variance as the residuals, but without any trend in them, and compute τ the same way.
  p = share of surrogates with τ ≥ the observed τ. This is stricter than shuffling: monthly residuals are
  autocorrelated, which makes rolling indicators smooth and gives large τ values by chance.

In [ ]:
def detrend_gaussian(x, bandwidth_frac):
    sigma = max(1.0, bandwidth_frac * len(x))
    trend = gaussian_filter1d(x.values.astype(float), sigma=sigma, mode="nearest")
    return pd.Series(x.values - trend, index=x.index), pd.Series(trend, index=x.index)


def indicators(resid, window):
    s = pd.Series(np.asarray(resid, dtype=float))
    return pd.DataFrame({
        "ar1": s.rolling(window).corr(s.shift(1)),
        "sd": s.rolling(window).std(),
        "skew": s.rolling(window).skew(),
        "kurtosis": s.rolling(window).kurt(),
    })


def kendall(values):
    v = np.asarray(values, dtype=float)
    mask = ~np.isnan(v)
    if mask.sum() < 10:
        return np.nan
    return kendalltau(np.arange(len(v))[mask], v[mask])[0]


def ar1_surrogates(resid, n, rng):
    # series with the same lag-1 autocorrelation and SD as resid, but stationary (no trend in AR1/SD)
    x = np.asarray(resid, dtype=float)
    phi = np.clip(pd.Series(x).autocorr(1), -0.99, 0.99)
    sigma_e = np.std(x) * np.sqrt(1 - phi ** 2)
    sims = np.empty((n, len(x)))
    sims[:, 0] = rng.normal(0, np.std(x), n)
    eps = rng.normal(0, sigma_e, (n, len(x)))
    for t in range(1, len(x)):
        sims[:, t] = phi * sims[:, t - 1] + eps[:, t]
    return sims


def ews_segment(pid, end=None):
    x = deseason[pid] if end is None else deseason[pid].loc[:end - pd.DateOffset(months=1)]
    filled_frac = 1 - observed[pid].reindex(x.index).mean()
    if len(x) < CONFIG["min_segment_months"] or filled_frac > CONFIG["max_filled_frac"]:
        return None
    resid, smooth = detrend_gaussian(x, CONFIG["bandwidth_frac"])
    window = max(12, int(CONFIG["window_frac"] * len(x)))
    ind = indicators(resid, window)
    ind.index = x.index

    out = {"n_months": len(x), "filled_frac": round(filled_frac, 2), "window_months": window}
    sims = ar1_surrogates(resid, CONFIG["n_surrogates"], rng)
    sim_taus = np.array([[kendall(c) for c in indicators(sim, window).T.values] for sim in sims])
    for j, key in enumerate(ind.columns):
        tau = kendall(ind[key])
        out[f"tau_{key}"] = round(tau, 3)
        out[f"p_{key}"] = round(float(np.mean(sim_taus[:, j] >= tau)), 3)

    # check: does the number of images per month change in the same period? More images -> higher and
    # less noisy monthly max, which can create a trend in SD that has nothing to do with resilience.
    out["tau_n_images"] = round(kendall(n_images[pid].reindex(x.index).values), 3)
    return out, x, smooth, resid, ind

## 5. Run the EWS

Takes a few minutes (surrogates). Points whose segment is too short or has too many interpolated months are skipped.

In [ ]:
rows, details = [], {}
controls_run = (control_points if len(control_points) <= CONFIG["max_controls"]
                else list(rng.choice(control_points, CONFIG["max_controls"], replace=False)))
for group, pids in [("pre-loss", list(loss_points)), ("control", controls_run)]:    
    for pid in pids:
        end = loss_points[pid]["window_start"] if group == "pre-loss" else None
        r = ews_segment(pid, end)
        if r is None:
            continue
        out, x, smooth, resid, ind = r
        rows.append({"point_id": pid, "group": group, "traj": point_traj.get(pid),
                     "loss_start": end, **out})
        details[pid] = (x, smooth, resid, ind)

ews = pd.DataFrame(rows)
a = CONFIG["alpha"]
ews["rising_ar1"] = (ews["tau_ar1"] > 0) & (ews["p_ar1"] < a)
ews["rising_sd"] = (ews["tau_sd"] > 0) & (ews["p_sd"] < a)
ews["ews_signal"] = ews["rising_ar1"] & ews["rising_sd"]       # both rising = strongest evidence
ews.to_csv(os.path.join(CONFIG["output_dir"], "ews_results.csv"), index=False)
print(ews["group"].value_counts().to_string())
skipped = len(loss_points) - (ews["group"] == "pre-loss").sum()
print(f"{skipped} loss points skipped (pre-loss segment < {CONFIG['min_segment_months']} months "
      f"or > {CONFIG['max_filled_frac']:.0%} interpolated)")

## 6. Plots per pre-loss point

In [ ]:
for pid in ews.loc[ews["group"] == "pre-loss", "point_id"]:
    x, smooth, resid, ind = details[pid]
    r = ews.set_index("point_id").loc[pid]
    full = deseason[pid]

    fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
    axes[0].plot(full.index, full, color="lightgray", lw=0.8, label="deseasonalised NDVI (full)")
    axes[0].plot(x.index, x, color="tab:green", lw=1, label="pre-loss segment")
    axes[0].plot(smooth.index, smooth, color="black", lw=1.5, label="Gaussian trend")
    axes[0].axvline(r["loss_start"], color="tab:red", ls="--", label=f"loss starts {r['loss_start']:%Y-%m}")
    axes[0].set_ylabel("NDVI"); axes[0].legend(fontsize=7, loc="lower left")
    axes[1].plot(resid.index, resid, color="tab:gray", lw=0.8); axes[1].set_ylabel("residual")
    axes[2].plot(ind.index, ind["ar1"], color="tab:blue")
    axes[2].set_ylabel("AR1"); axes[2].set_title(f"AR1: τ = {r['tau_ar1']:+.2f}, p = {r['p_ar1']:.3f}", fontsize=9)
    axes[3].plot(ind.index, ind["sd"], color="tab:orange")
    axes[3].set_ylabel("SD"); axes[3].set_title(f"SD: τ = {r['tau_sd']:+.2f}, p = {r['p_sd']:.3f}"
                                                f"   (n_images τ = {r['tau_n_images']:+.2f})", fontsize=9)
    fig.suptitle(f"Point {pid} ")
    fig.tight_layout()
    fig.savefig(os.path.join(CONFIG["output_dir"], f"ews_{pid}.png"), dpi=120)
    plt.show()

## 7. Results table

In [ ]:
cols = ["point_id", "group", "traj", "loss_start", "n_months", "filled_frac",
        "tau_ar1", "p_ar1", "tau_sd", "p_sd", "tau_skew", "p_skew", "tau_kurtosis", "p_kurtosis",
        "tau_n_images", "rising_ar1", "rising_sd", "ews_signal"]
ews.sort_values(["group", "point_id"], ascending=[False, True])[cols].reset_index(drop=True)

## 8. Summary: hits vs. false alarms

An EWS is only useful if it shows up **more often before a loss than in points without one**.
The control group gives the false-alarm rate.

In [ ]:
summary = ews.groupby("group")[["rising_ar1", "rising_sd", "ews_signal"]].mean().mul(100).round(1)
summary.insert(0, "n_points", ews["group"].value_counts())
summary = summary.reindex(["pre-loss", "control"])
display(summary)

fig, ax = plt.subplots(figsize=(7, 4))
xpos = np.arange(3)
labels = ["rising AR1", "rising SD", "both"]
for i, (group, color) in enumerate([("pre-loss", "tab:red"), ("control", "tab:gray")]):
    if group in summary.index:
        vals = summary.loc[group, ["rising_ar1", "rising_sd", "ews_signal"]].values
        ax.bar(xpos + (i - 0.5) * 0.38, vals, 0.38, color=color,
               label=f"{group} (n={int(summary.loc[group, 'n_points'])})")
ax.set_xticks(xpos); ax.set_xticklabels(labels)
ax.set_ylabel(f"% of points (p < {CONFIG['alpha']})")
ax.set_title("Early warning signals: before a loss vs. controls")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(CONFIG["output_dir"], "ews_hits_vs_controls.png"), dpi=120)
plt.show()

## new section; scenario breakdown  

In [ ]:
# Martinez et al. (2024) scenario matrix: NDVI trend x SD trend
SCENARIO_LABELS = {
    1: "Abrupt Transition (potential EWS)",
    2: "Slowly Transitioning (no EWS)",
    3: "Recovering",
    4: "Stable",
}

def classify_scenario(ndvi_tau, sd_tau):
    # only the SIGN of both taus decides the quadrant (no p-values)
    if pd.isna(ndvi_tau) or pd.isna(sd_tau) or ndvi_tau == 0 or sd_tau == 0:
        return None
    if ndvi_tau < 0:
        return 1 if sd_tau > 0 else 2
    return 3 if sd_tau > 0 else 4

# NDVI trend over the SAME segment as the indicators (pre-loss segment, or full series for controls)
ews["tau_ndvi"] = [round(kendall(details[pid][0].values), 3) for pid in ews["point_id"]]
ews["scenario"] = [classify_scenario(n, s) for n, s in zip(ews["tau_ndvi"], ews["tau_sd"])]
ews["scenario_label"] = ews["scenario"].map(SCENARIO_LABELS)
ews.to_csv(os.path.join(CONFIG["output_dir"], "ews_results.csv"), index=False)

# Breakdown per group; "with p_sd < alpha" = the SD trend is also significant
for group in ["pre-loss", "control"]:
    g = ews[ews["group"] == group]
    print(f"\n{group} (n={len(g)}):")
    for s, label in SCENARIO_LABELS.items():
        m = g["scenario"] == s
        n_sig = (m & (g["p_sd"] < CONFIG["alpha"])).sum()
        print(f"  {s} {label:36s} {m.sum():4d} ({100 * m.mean():3.0f}%)   with p_sd < {CONFIG['alpha']}: {n_sig}")

# Quadrant plot
fig, ax = plt.subplots(figsize=(9, 7))
for group, color in [("control", "tab:gray"), ("pre-loss", "tab:red")]:
    g = ews[ews["group"] == group]
    sig = g["p_sd"] < CONFIG["alpha"]
    ax.scatter(g.loc[~sig, "tau_ndvi"], g.loc[~sig, "tau_sd"], color=color, s=40, alpha=0.4,
               edgecolor="black", linewidth=0.3, label=f"{group}, SD trend n.s.")
    ax.scatter(g.loc[sig, "tau_ndvi"], g.loc[sig, "tau_sd"], color=color, s=60, alpha=0.9,
               edgecolor="black", linewidth=0.8, label=f"{group}, SD trend p < {CONFIG['alpha']}")
ax.axhline(0, color="gray", linewidth=1, zorder=1)
ax.axvline(0, color="gray", linewidth=1, zorder=1)
ax.text(0.02, 0.98, "1: Abrupt Transition\n(potential EWS)", transform=ax.transAxes, ha="left", va="top", fontsize=9, color="dimgray")
ax.text(0.02, 0.02, "2: Slowly Transitioning\n(no EWS)", transform=ax.transAxes, ha="left", va="bottom", fontsize=9, color="dimgray")
ax.text(0.98, 0.98, "3: Recovering", transform=ax.transAxes, ha="right", va="top", fontsize=9, color="dimgray")
ax.text(0.98, 0.02, "4: Stable", transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="dimgray")
ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
ax.set_xlabel("NDVI trend (Kendall tau)")
ax.set_ylabel("Rolling SD trend (Kendall tau)")
ax.set_title("Martinez et al. (2024) scenario matrix (monthly)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(CONFIG["output_dir"], "scenario_matrix.png"), dpi=120)
plt.show()

In [ ]:
# tau_cols = ['tau_ndvi', 'tau_ar1', 'tau_sd', 'tau_skew', 'tau_kurtosis']
# traj_colors = {1: '#12301f', 2: '#9fbf6a', 3: '#e31a1c', 4: '#1f78b4', 5: '#ffd92f'}

# n = len(tau_cols)
# fig, axes = plt.subplots(n, n, figsize=(14, 14))
# for i, yvar in enumerate(tau_cols):
#     for j, xvar in enumerate(tau_cols):
#         ax = axes[i, j]
#         if i == j:
#             for t, group in results_df.groupby('traj'):
#                 ax.hist(group[xvar].dropna(), color=traj_colors.get(t, 'gray'), alpha=0.5, bins=15)
#         else:
#             for t, group in results_df.groupby('traj'):
#                 ax.scatter(group[xvar], group[yvar], color=traj_colors.get(t, 'gray'), s=15, alpha=0.7)
#             ax.set_ylim(-1, 1)
#         ax.set_xlim(-1, 1)
#         if i == n - 1:
#             ax.set_xlabel(xvar, fontsize=8)
#         if j == 0:
#             ax.set_ylabel(yvar, fontsize=8)
#         ax.tick_params(labelsize=6)
# fig.suptitle('All pairwise tau combinations, colored by trajectory class', y=1.01)
# fig.tight_layout()
# plt.show()

In [ ]:
# 1. Why only 8 pre-loss points? (how many losses were skipped for being too short / too interpolated)
print(len(loss_points), "loss points in total,", (ews["group"] == "pre-loss").sum(), "with enough data for EWS")

# 2. Is the falling SD explained by more images per month?
print(ews.groupby("group")[["tau_sd", "tau_n_images"]].median())
print("correlation tau_sd vs tau_n_images:", ews["tau_sd"].corr(ews["tau_n_images"]).round(2))

In [ ]:
# Sensitivity of the EWS to window size and detrending bandwidth (no surrogates -> fast)
window_fracs = [0.25, 0.375, 0.5, 0.625, 0.75]
bandwidth_fracs = [0.1, 0.15, 0.2, 0.25, 0.3]

def taus(pid, end, wf, bf):
    x = deseason[pid] if end is None else deseason[pid].loc[:end - pd.DateOffset(months=1)]
    if len(x) < CONFIG["min_segment_months"] or 1 - observed[pid].reindex(x.index).mean() > CONFIG["max_filled_frac"]:
        return None
    resid, _ = detrend_gaussian(x, bf)
    ind = indicators(resid, max(12, int(wf * len(x))))
    return kendall(ind["ar1"]), kendall(ind["sd"])

groups = {"pre-loss": [(p, b["window_start"]) for p, b in loss_points.items()],
          "control": [(p, None) for p in control_points]}
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for row, (group, items) in enumerate(groups.items()):
    grid = np.full((2, len(bandwidth_fracs), len(window_fracs)), np.nan)
    for i, bf in enumerate(bandwidth_fracs):
        for j, wf in enumerate(window_fracs):
            t = [taus(p, end, wf, bf) for p, end in items]
            t = np.array([v for v in t if v is not None])
            if len(t):
                grid[:, i, j] = np.nanmedian(t, axis=0)
    for col, name in enumerate(["AR1", "SD"]):
        ax = axes[row, col]
        im = ax.imshow(grid[col], cmap="RdBu_r", vmin=-1, vmax=1, origin="lower")
        for i in range(len(bandwidth_fracs)):
            for j in range(len(window_fracs)):
                ax.text(j, i, f"{grid[col, i, j]:+.2f}", ha="center", va="center", fontsize=8)
        ax.set_xticks(range(len(window_fracs))); ax.set_xticklabels(window_fracs)
        ax.set_yticks(range(len(bandwidth_fracs))); ax.set_yticklabels(bandwidth_fracs)
        ax.set_xlabel("window_frac"); ax.set_ylabel("bandwidth_frac")
        ax.set_title(f"{group}: median Kendall tau of {name}")
fig.colorbar(im, ax=axes, shrink=0.6, label="median tau")
fig.savefig(os.path.join(CONFIG["output_dir"], "ews_sensitivity.png"), dpi=120)
plt.show()

In [ ]:

df = pd.read_csv("ews_output_monthly/ews_results.csv")
df = df[df["group"]=="pre-loss"]
EWS_points = df[df["scenario"]==1]
print(EWS_points)


In [ ]:
##### CHANGE THIS SO IT IS CORRECT, NOW JUST COPY PASTE FROM OTHER NOTEBOOK

# table = breakpoints_df.sort_values("breakpoint_date").copy()
# table["class"] = table["traj"].map(traj_labels)
# table["breakpoint_year"] = table["breakpoint_date"].dt.year
# table = table[["point_id", "traj", "class", "type", "breakpoint_date", "breakpoint_year", "magnitude"]]
# table.to_csv(os.path.join(CONFIG["output_dir"], "breakpoints.csv"), index=False)
# table.reset_index(drop=True)

# breakpoints = pd.read_csv("ndvi_analysis_output_monthly/breakpoints.csv")
# breakpoints_drop = breakpoints[breakpoints['type'] == 'drop']
# print(breakpoints_drop)
# breakpoints_drop.to_csv("ndvi_analysis_output_monthly/breakpoints_drop.csv", index=False)